In [83]:
import pandas as pd 
from sqlalchemy import create_engine, text, Text
from sqlalchemy.dialects.postgresql import ARRAY
from dotenv import load_dotenv
import os

In [84]:
load_dotenv()
POSTGRES_URI = os.getenv("POSTGRES_URI")
engine = create_engine(POSTGRES_URI)

In [85]:
with engine.begin() as conn:
    df = pd.read_sql(
        "raw_netflix_titles",
        con=conn,
        index_col="show_id"
    )
df.head()

,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
show_id,,,,,,,,,,,
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
s4,TV Show,Jailbirds New Orleans,None,None,None,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [86]:
df["cast"] =  df["cast"].map(lambda x: str(x).split(","))
df["cast"] = df["cast"].map(lambda x : [items.strip() for items in x])


In [87]:
df['listed_in'] =  df['listed_in'].map(lambda x: str(x).split(","))
df['listed_in'] = df['listed_in'].map(lambda x: [item.strip() for item in x])


In [88]:
with engine.begin() as conn:
    query = text("""
                    CREATE TABLE IF NOT EXISTS netflix_titles(
                    show_id VARCHAR(20) PRIMARY KEY NOT NULL,
                    type TEXT NOT NULL,
                    title TEXT NOT NULL,
                    director TEXT,
                    "cast" TEXT[],
                    country TEXT,
                    date_added TEXT, 
                    release_year INT,
                    rating TEXT,
                    duration TEXT,
                    listed_in TEXT[], 
                    description TEXT
                )
                """)
    conn.execute(query)

    df.to_sql(
        con=conn,
        name='netflix_titles',
        if_exists='replace',
        index_label='show_id',
        dtype={
            "cast" : ARRAY(Text),
            "listed_in" : ARRAY(Text)
        }
    )